In [0]:
%pip install databricks-langchain==0.12.1 langchain-community==0.4.1 langchain-experimental==0.4.1

In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from databricks_langchain import ChatDatabricks
import os

In [0]:
!pip install python-dotenv

In [0]:
from dotenv import load_dotenv
import os
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")



# Set environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

In [0]:
llm = ChatDatabricks(endpoint=  "databricks-meta-llama-3-3-70b-instruct",
    temperature= 0.1,
    max_tokens = 250,
)


In [0]:
topic_builder_agent_prompt_template = PromptTemplate(
    input_variables = ["user_query"],
    template = """ You are a topic builder agent
    The first tep in a sequential workflow
    Your task is to take the users learnng goals and produce  structured outputs which can be used by subsequent workflows. Topics should be aligned with microsoft techstack
    Output format should be as follows:
     User Goal : ummary of the users goal
     Topics
     -Topic 1
      -Subtopic 1.1
       -Subtopic 1.2
       user query is {user_query}"""
)

In [0]:
module_picker_agent_prompt_template = PromptTemplate(
    input_variables = ["topics"],
    template = """
    You are a MS learn module picker agent, second step in sequential workflow
    You receive structured outputs from topic builder agent
    Your job is to map each topic and subtopic to most relevant Microsoft Learning Modules
    Returm a structured list of topics with their corresponding MS learning modules and leaening paths,
    Return a structured list of topics with urlS AND BRIEF DESCRIPTON.
    THE TOPICS ARE {topics}
    """
)

In [0]:
study_plan_generator_prompt_template = PromptTemplate(
    input_variables = ["modules"],
    template = """ You are a study plan agent, The third step in the sequental flow
    You receive MS lEARN mODULE AND USERS TIMEline information
    Yur job is to convert this to a realistic week by week study plan.
    Your output must be clean and human friendly.
    The modules are {modules}
    """
)



In [0]:
topic_generator_chain = topic_builder_agent_prompt_template|llm
module_selection_chain = module_picker_agent_prompt_template|llm
study_plan_chain = study_plan_generator_prompt_template|llm
ms_learn_path_builder = RunnableSequence(topic_generator_chain,module_selection_chain,study_plan_chain)


In [0]:
result = ms_learn_path_builder.invoke(
    {
        "user_query": "I want to become an AI engineer in 3 Months"
    }
)

print("Result \n")
print(result.content)